# EVE 310 - Lab 06: Regression features: outliers, scaling, dummies

**Module 2 | 10/01/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab06-regression-features/notebooks/lab06-tutorial.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Remove outliers more than 3 standard deviations from the mean
2. Add an interaction term and min-max normalize numeric columns
3. Create dummy variables with `pd.get_dummies` and fit a time-aware linear model

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It creates `DATA_DIR` and `FIGURES_DIR` and downloads this lab's data files.
3. Work down the notebook in order. Cells marked **Your turn** are the ones you complete.
4. Nothing to submit for this notebook. The **activity** notebook is the one you download as `.ipynb` and upload to Gradescope.


Extension of Lab 5 using daily energy use in Jester East (`eU`). You will clean outliers, add an interaction, normalize, dummy-code categoricals, then fit a linear model and plot a short time window.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab06-regression-features"
DATA_FILES = ["ECJ_Energy_Lab6Activity.csv", "JES_Energy_Lab6Tutorial.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

import datetime as dt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

## 1. Load the data


In [ ]:
e_df = pd.read_csv(DATA_DIR / 'JES_Energy_Lab6Tutorial.csv')
print(e_df.head())
e_df.info()
e_df['allDate'] = pd.to_datetime(e_df['allDate'])
e_df.info()


## 2. Worked example — outliers

A quick plot of `eU` shows spikes. Treat points more than 3 standard deviations from the mean as outliers.


In [ ]:
fig, ax = plt.subplots()
ax.plot(e_df['allDate'], e_df['eU'])
ax.set_xlabel('Date')
ax.set_ylabel('Energy use')
fig.savefig(FIGURES_DIR / 'lab06_eu_raw.png', dpi=200)

n_before = len(e_df)
eu_std = np.std(e_df['eU'], ddof=1)
eu_mean = np.mean(e_df['eU'])
upper = eu_mean + 3 * eu_std
lower = eu_mean - 3 * eu_std
e_df = e_df[(e_df['eU'] < upper) & (e_df['eU'] > lower)]
print(f'Removed {n_before - len(e_df)} rows; {len(e_df)} remain')


Interaction term, then min-max normalize the numeric columns so values lie in [0, 1].


In [ ]:
e_df['Temp_x_Med'] = e_df['avgTemp'] * e_df['MonMed']

for col in ['eU', 'MonMed', 'avgTemp', 'Temp_x_Med']:
    lo, hi = e_df[col].min(), e_df[col].max()
    e_df[col] = (e_df[col] - lo) / (hi - lo)

fig, ax = plt.subplots()
ax.plot(e_df['allDate'], e_df['eU'])
ax.set_xlabel('Date')
ax.set_ylabel('Normalized energy consumption')


`pd.get_dummies(..., drop_first=True)` encodes categoricals and drops one level per variable.


In [ ]:
e_df_dummy = pd.get_dummies(
    e_df, columns=['classSession', 'dayType', 'season'], drop_first=True
)
e_df_dummy.head()


In [ ]:
y = e_df_dummy['eU']
x = e_df_dummy.drop(columns=['eU'])
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.30, random_state=42
)
x_train_date = x_train['allDate']
x_test_date = x_test['allDate']
x_train = x_train.drop(columns=['allDate'])
x_test = x_test.drop(columns=['allDate'])

lm = LinearRegression()
lm.fit(x_train, y_train)
y_train_predicted = lm.predict(x_train)
y_test_predicted = lm.predict(x_test)
print('train R2', r2_score(y_train, y_train_predicted))
print('test  R2', r2_score(y_test, y_test_predicted))


In [ ]:
fig, ax = plt.subplots()
ax.plot(y_train, y_train_predicted, ls='', marker='o')
ax.plot([0, 1], [0, 1], ls='--', alpha=0.7)
ax.set_xlabel('Observed energy consumption')
ax.set_ylabel('Predicted energy consumption')
fig.savefig(FIGURES_DIR / 'lab06_pred_vs_obs.png', dpi=200)


Time-series view for 1 Jan–1 Mar 2014 (training points only):


In [ ]:
fig, ax = plt.subplots()
ax.plot(x_train_date, y_train, ls='', marker='o', label='Actual')
ax.plot(x_train_date, y_train_predicted, ls='', marker='o', label='Predicted')
fig.autofmt_xdate()
ax.set_xlim(dt.date(2014, 1, 1), dt.date(2014, 3, 1))
ax.set_xlabel('Date')
ax.set_ylabel('Normalized energy consumption')
ax.legend()
fig.savefig(FIGURES_DIR / 'lab06_timeseries.png', dpi=200)


## 3. Wrap-up

Continue with `lab06-activity.ipynb` using the ECJ energy file. That activity **standardizes** (z-score) instead of min-max normalizing.
